# Phase 3 - embedding precompute (Colab GPU)

Runs `src/data.py` then `src/embed.py` to encode every dataset sentence once
with the frozen `all-mpnet-base-v2` backbone.

**Storage:** `.npz` files are written to Colab's fast local disk (`embeddings/`),
then `rsync`ed to a **Google Drive** folder after each dataset. `nli/train` is
checkpointed per 100k-sentence chunk, so an OOM / disconnect resumes from the
last chunk rather than restarting the split.

Runtime: **Runtime -> Change runtime type -> T4 GPU**. Full run ~25-35 min.

Everything after this (features, heads, pilot, grid, analysis) runs on CPU off
the cached `.npz` files.

In [ ]:
!git clone https://github.com/ryanteachman/sbert-head-ablation.git
%cd sbert-head-ablation
!pip install -q "datasets==4.0.0" hf_xet "sentence-transformers==5.1.2" pyarrow pyyaml scikit-learn
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_EMB = '/content/drive/MyDrive/sbert-head-ablation/embeddings'   # persistent
import os; os.makedirs(DRIVE_EMB, exist_ok=True)
# pull any prior progress from Drive to local disk
!mkdir -p embeddings && rsync -a "{DRIVE_EMB}/" embeddings/
!find embeddings -name '*.npz' | sort; echo '---'; cat embeddings/meta.json 2>/dev/null || echo '(no meta yet)'

## 1. Build processed splits
Deterministic (fixed QQP split seed). Verifies every split size against
`PROTOCOL.md` section 5.

In [ ]:
!python src/data.py

## 2. Sanity-check the encoder
Confirms L2-normalized + deterministic output before the real run.

In [ ]:
!python src/embed.py --verify

## 3. Encode, one dataset per cell
Each cell encodes to local disk then syncs that dataset to Drive.

**NLI is the long one** (`nli/train` ~1.15M sentences, 12 chunks, ~20-25 min on
a T4). If it OOMs or disconnects, just re-run the cell - it resumes from the
last finished chunk. During `saving ...` / `rsync` the cell can look idle;
let it finish. If the GPU still OOMs, add `--encode-batch 128`.

In [ ]:
!python src/embed.py --datasets nli && rsync -a --info=progress2 --exclude='_wip_*' embeddings/ "{DRIVE_EMB}/"

In [ ]:
!python src/embed.py --datasets qqp && rsync -a --info=progress2 --exclude='_wip_*' embeddings/ "{DRIVE_EMB}/"

In [ ]:
!python src/embed.py --datasets paws && rsync -a --info=progress2 --exclude='_wip_*' embeddings/ "{DRIVE_EMB}/"

## 4. Inspect the cache

In [ ]:
import json, numpy as np, pathlib
meta = json.load(open('embeddings/meta.json'))
print(json.dumps(meta, indent=2))
print()
tot = 0
for p in sorted(pathlib.Path('embeddings').rglob('*.npz')):
    with np.load(p) as z:
        tot += p.stat().st_size
        print(f'{str(p.relative_to("embeddings")):<32} uniq_emb{z["uniq_emb"].shape} {z["uniq_emb"].dtype}  '
              f'pairs={len(z["label"]):>9,}  {p.stat().st_size/1e6:6.0f} MB')
print(f'\n{len(meta["splits"])}/11 splits cached,  {tot/1e9:.2f} GB total')
assert len(meta['splits']) == 11, 'not all splits done - re-run the encode cells'
!rsync -a --exclude='_wip_*' embeddings/ "{DRIVE_EMB}/"
print('synced to', DRIVE_EMB)

## Done
All 11 `.npz` + `meta.json` are in `MyDrive/sbert-head-ablation/embeddings/`.
Next: `notebooks/pilot_colab.ipynb`.